In [ ]:
!pip install -q plotly==5.24.1 kaleido==0.2.1 openpyxl itables matplotlib pandas

In [ ]:
import plotly
import kaleido
print("plotly version:", plotly.__version__)
print("kaleido version:", kaleido.__version__)

plotly version: 5.24.1
kaleido version: 0.2.1


In [ ]:
# ============================================
# CELL 3 — Imports and data loading
# ============================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display

file_path = "/content/Hoh_detections_per_minute_20251103.xlsx"

df = pd.read_excel(file_path)

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["hour"] = pd.to_numeric(df["hour"], errors="coerce")
df["minute"] = pd.to_numeric(df["minute"], errors="coerce")
df["detections"] = pd.to_numeric(df["detections"], errors="coerce")

df = df.dropna(subset=["species", "hour", "detections"]).copy()

df["hour"] = df["hour"].astype(int)
if "aru_number" in df.columns:
    df["aru_number"] = df["aru_number"].astype(str)

df.loc[df["detections"] < 0, "detections"] = 0

print("Data loaded successfully.")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Data loaded successfully.
Shape: (236944, 7)
Columns: ['study_site', 'aru_number', 'date', 'hour', 'minute', 'species', 'detections']


,study_site,aru_number,date,hour,minute,species,detections
0,Hoh,0,2025-05-08,0,0,Northern Saw-whet Owl,19
1,Hoh,0,2025-05-08,0,1,Northern Saw-whet Owl,20
2,Hoh,0,2025-05-08,0,2,Northern Saw-whet Owl,20
3,Hoh,0,2025-05-08,0,3,Northern Saw-whet Owl,19
4,Hoh,0,2025-05-08,0,4,Common Raven,1


In [ ]:
# ============================================
# CELL 4 — Helper functions for export
# ============================================
def export_plotly(fig, png_name, html_name, width=1400, height=850, scale=3):
    fig.write_image(png_name, width=width, height=height, scale=scale)
    fig.write_html(
        html_name,
        full_html=True,
        include_plotlyjs=True,
        config={
            "displayModeBar": True,
            "scrollZoom": True,
            "responsive": True
        }
    )
    print(f"Saved: {png_name}")
    print(f"Saved: {html_name}")

def export_table_png(df_table, filename="table.png", title=None, figsize=(14, 8), fontsize=11):
    fig, ax = plt.subplots(figsize=figsize)
    ax.axis("off")

    if title:
        plt.title(title, fontsize=16, pad=18, weight="bold")

    table = ax.table(
        cellText=df_table.values,
        colLabels=df_table.columns,
        loc="center",
        cellLoc="center"
    )

    table.auto_set_font_size(False)
    table.set_fontsize(fontsize)
    table.scale(1, 1.6)

    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight="bold", color="black")
            cell.set_facecolor("#DCEAF7")
        else:
            if row % 2 == 0:
                cell.set_facecolor("#F7FBFF")
            else:
                cell.set_facecolor("white")

    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved: {filename}")

In [ ]:
# ============================================
# CELL 4 — Figure 1: Total detections by hour
# ============================================
hourly_total = (
    df.groupby("hour", as_index=False)["detections"]
      .sum()
      .sort_values("hour")
)

# 补齐 0-23 小时
all_hours = pd.DataFrame({"hour": list(range(24))})
hourly_total = all_hours.merge(hourly_total, on="hour", how="left").fillna(0)

peak_row = hourly_total.loc[hourly_total["detections"].idxmax()]
peak_hour = int(peak_row["hour"])
peak_value = float(peak_row["detections"])

fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=hourly_total["hour"],
    y=hourly_total["detections"],
    mode="lines+markers",
    line=dict(width=4, shape="spline", smoothing=1.0),
    marker=dict(size=8),
    fill="tozeroy",
    hovertemplate="<b>%{x}:00</b><br>Total detections: %{y:,.0f}<extra></extra>",
    name="Total detections"
))

fig1.add_trace(go.Scatter(
    x=[peak_hour],
    y=[peak_value],
    mode="markers+text",
    marker=dict(size=16, symbol="star"),
    text=[f"Peak: {peak_hour}:00"],
    textposition="top center",
    hovertemplate=f"<b>Peak hour</b><br>{peak_hour}:00<br>Detections: {peak_value:,.0f}<extra></extra>",
    name="Peak"
))

fig1.update_layout(
    title=dict(
        text="Hoh Audio Detections Across the Day",
        x=0.5,
        xanchor="center"
    ),
    xaxis=dict(
        title="Hour of day",
        tickmode="array",
        tickvals=list(range(24)),
        ticktext=[f"{h}:00" for h in range(24)],
        showgrid=True
    ),
    yaxis=dict(
        title="Total detections",
        showgrid=True,
        separatethousands=True
    ),
    template="plotly_white",
    height=520,
    width=1000,
    hovermode="x unified"
)

fig1.add_annotation(
    x=peak_hour,
    y=peak_value,
    text=f"Highest activity at {peak_hour}:00<br>{peak_value:,.0f} detections",
    showarrow=True,
    arrowhead=2,
    ax=0,
    ay=-60
)

fig1.show()

export_plotly(
    fig1,
    "01_hoh_total_detections_by_hour.png",
    "01_hoh_total_detections_by_hour.html",
    width=1500,
    height=850,
    scale=3
)

Saved: 01_hoh_total_detections_by_hour.png
Saved: 01_hoh_total_detections_by_hour.html


In [ ]:
# ============================================
# CELL 5 — Figure 2: Heatmap of top species across hours
# ============================================
top_n_species = 10

top_species_df = (
    df.groupby("species", as_index=False)["detections"]
      .sum()
      .sort_values("detections", ascending=False)
      .head(top_n_species)
)

top_species = top_species_df["species"].tolist()

heat_df = df[df["species"].isin(top_species)].copy()

heat_agg = (
    heat_df.groupby(["species", "hour"], as_index=False)["detections"]
           .sum()
)

heat_pivot = heat_agg.pivot(index="species", columns="hour", values="detections").fillna(0)

# 依照总量排序
species_order = (
    heat_df.groupby("species")["detections"]
           .sum()
           .sort_values(ascending=False)
           .index.tolist()
)
heat_pivot = heat_pivot.loc[species_order]

# 补齐 0-23
for h in range(24):
    if h not in heat_pivot.columns:
        heat_pivot[h] = 0
heat_pivot = heat_pivot[sorted(heat_pivot.columns)]

species_totals = (
    heat_df.groupby("species")["detections"]
           .sum()
           .to_dict()
)

y_labels = [f"{sp} ({int(species_totals[sp]):,})" for sp in heat_pivot.index]

# 用 log1p 让低值也能看见
z_values = np.log1p(heat_pivot.values)

fig2 = go.Figure(data=go.Heatmap(
    z=z_values,
    x=[f"{h}:00" for h in heat_pivot.columns],
    y=y_labels,
    customdata=heat_pivot.values,
    colorscale="Viridis",
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Hour: %{x}<br>"
        "Detections: %{customdata:,.0f}<br>"
        "log(1 + detections): %{z:.2f}<extra></extra>"
    ),
    colorbar=dict(title="log(1 + detections)")
))

fig2.update_layout(
    title=dict(
        text=f"Hoh Soundscape by Time of Day Top {top_n_species} Species",
        x=0.5,
        xanchor="center"
    ),
    xaxis_title="Hour of day",
    yaxis_title="Species (total detections)",
    template="plotly_white",
    height=750,
    width=1150
)

fig2.show()

export_plotly(
    fig2,
    "02_hoh_heatmap_top_species.png",
    "02_hoh_heatmap_top_species.html",
    width=1700,
    height=1100,
    scale=3
)

Saved: 02_hoh_heatmap_top_species.png
Saved: 02_hoh_heatmap_top_species.html


In [ ]:
# ============================================
# CELL 6 — Table 3: Species list / summary
# ============================================
species_summary = (
    df.groupby("species")
      .agg(
          total_detections=("detections", "sum"),
          mean_detections=("detections", "mean"),
          max_single_minute=("detections", "max"),
          active_rows=("detections", lambda x: (x > 0).sum()),
          n_hours_present=("hour", "nunique"),
          n_dates_present=("date", "nunique"),
          n_aru_present=("aru_number", "nunique") if "aru_number" in df.columns else ("species", "count")
      )
      .reset_index()
)

species_hour = (
    df.groupby(["species", "hour"], as_index=False)["detections"]
      .sum()
)

idx = species_hour.groupby("species")["detections"].idxmax()
species_peak_hour = (
    species_hour.loc[idx, ["species", "hour", "detections"]]
    .rename(columns={
        "hour": "peak_hour",
        "detections": "peak_hour_detections"
    })
)

species_summary = species_summary.merge(species_peak_hour, on="species", how="left")

species_summary = species_summary.sort_values(
    by=["total_detections", "n_hours_present"],
    ascending=[False, False]
).reset_index(drop=True)

species_summary["mean_detections"] = species_summary["mean_detections"].round(2)
species_summary["peak_hour_label"] = species_summary["peak_hour"].astype(int).astype(str) + ":00"

# 导出完整 CSV
species_summary_export = species_summary[[
    "species",
    "total_detections",
    "mean_detections",
    "max_single_minute",
    "active_rows",
    "n_hours_present",
    "n_dates_present",
    "n_aru_present",
    "peak_hour_label",
    "peak_hour_detections"
]].copy()

species_summary_export.columns = [
    "Species",
    "Total detections",
    "Mean detections",
    "Max single-minute detections",
    "Active rows",
    "Hours present",
    "Dates present",
    "ARUs present",
    "Peak hour",
    "Peak-hour detections"
]

species_summary_export.to_csv("03_hoh_species_list_full.csv", index=False)
print("Saved: 03_hoh_species_list_full.csv")

# 导出前 20 作为 PPT 表格
pretty_species = species_summary_export[[
    "Species",
    "Total detections",
    "Hours present",
    "Dates present",
    "Peak hour",
    "Peak-hour detections"
]].copy().head(20)

pretty_species["Total detections"] = pretty_species["Total detections"].map(lambda x: f"{x:,.0f}")
pretty_species["Peak-hour detections"] = pretty_species["Peak-hour detections"].map(lambda x: f"{x:,.0f}")

display(pretty_species)

export_table_png(
    pretty_species,
    filename="03_hoh_species_list_top20.png",
    title="Hoh Species Summary (Top 20 by Total Detections)",
    figsize=(15, 9),
    fontsize=11
)

Saved: 03_hoh_species_list_full.csv


,Species,Total detections,Hours present,Dates present,Peak hour,Peak-hour detections
0,Pacific Wren,"210,064",17,49,5:00,"45,426"
1,Pacific-slope Flycatcher,"180,830",24,49,9:00,"20,924"
2,American Robin,"154,458",24,49,5:00,"64,144"
3,Golden-crowned Kinglet,"128,984",24,49,8:00,"9,834"
4,Swainson's Thrush,"115,183",21,49,21:00,"17,404"
5,Black-throated Gray Warbler,"53,375",17,49,9:00,"5,605"
6,Northern Saw-whet Owl,"35,338",24,48,23:00,"6,500"
7,Wilson's Warbler,"25,801",17,49,9:00,"3,042"
8,Varied Thrush,"19,649",22,50,5:00,"5,375"
9,Western Tanager,"17,532",23,49,5:00,"2,519"


Saved: 03_hoh_species_list_top20.png


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=pretty_species)

https://docs.google.com/spreadsheets/d/1RFZ3pePIk14OnD2GHER79b4-9sPIvitEejow8kWWCZY/edit#gid=0


In [ ]:
# ============================================
# CELL 7 — Figure 4: Top 10 species bar chart
# ============================================
top10 = (
    df.groupby("species", as_index=False)["detections"]
      .sum()
      .sort_values("detections", ascending=False)
      .head(10)
      .reset_index(drop=True)
)

top10_plot = top10.sort_values("detections", ascending=True)

fig4 = px.bar(
    top10_plot,
    x="detections",
    y="species",
    orientation="h",
    text="detections",
    title="Top 10 Species in Hoh by Total Detections",
    labels={"detections": "Total detections", "species": ""}
)

fig4.update_traces(
    texttemplate="%{text:,.0f}",
    textposition="outside",
    hovertemplate="<b>%{y}</b><br>Total detections: %{x:,.0f}<extra></extra>"
)

fig4.update_layout(
    template="plotly_white",
    height=650,
    width=1000,
    xaxis_title="Total detections",
    yaxis_title="",
    showlegend=False
)

fig4.show()

export_plotly(
    fig4,
    "04_hoh_top10_species_bar.png",
    "04_hoh_top10_species_bar.html",
    width=1500,
    height=950,
    scale=3
)

Saved: 04_hoh_top10_species_bar.png
Saved: 04_hoh_top10_species_bar.html


In [ ]:
# ============================================
# CELL 8 — Table 4: Top 10 species table
# ============================================
total_all = df["detections"].sum()

top10_table = top10.copy()
top10_table["share_of_all_detections_pct"] = (
    top10_table["detections"] / total_all * 100
).round(2)

top10_table.columns = [
    "Species",
    "Total detections",
    "Share of all detections (%)"
]

top10_table.to_csv("04_hoh_top10_species_table.csv", index=False)
print("Saved: 04_hoh_top10_species_table.csv")

top10_table_display = top10_table.copy()
top10_table_display["Total detections"] = top10_table_display["Total detections"].map(lambda x: f"{x:,.0f}")
top10_table_display["Share of all detections (%)"] = top10_table_display["Share of all detections (%)"].map(lambda x: f"{x:.2f}")

display(top10_table_display)

export_table_png(
    top10_table_display,
    filename="04_hoh_top10_species_table.png",
    title="Top 10 Species in Hoh",
    figsize=(13, 5.8),
    fontsize=11
)

Saved: 04_hoh_top10_species_table.csv


,Species,Total detections,Share of all detections (%)
0,Pacific Wren,"210,064",20.30
1,Pacific-slope Flycatcher,"180,830",17.47
2,American Robin,"154,458",14.92
3,Golden-crowned Kinglet,"128,984",12.46
4,Swainson's Thrush,"115,183",11.13
5,Black-throated Gray Warbler,"53,375",5.16
6,Northern Saw-whet Owl,"35,338",3.41
7,Wilson's Warbler,"25,801",2.49
8,Varied Thrush,"19,649",1.90
9,Western Tanager,"17,532",1.69


Saved: 04_hoh_top10_species_table.png


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=top10_table_display)

https://docs.google.com/spreadsheets/d/1bdBKBjQs6yYIPrkOMVwtsN7OwhaVYjfMEN1JxeouD7w/edit#gid=0


In [ ]:
# ============================================
# CELL 9 — Check exported files
# ============================================
import os

expected_files = [
    "01_hoh_total_detections_by_hour.png",
    "01_hoh_total_detections_by_hour.html",
    "02_hoh_heatmap_top_species.png",
    "02_hoh_heatmap_top_species.html",
    "03_hoh_species_list_top20.png",
    "03_hoh_species_list_full.csv",
    "04_hoh_top10_species_bar.png",
    "04_hoh_top10_species_bar.html",
    "04_hoh_top10_species_table.png",
    "04_hoh_top10_species_table.csv"
]

for f in expected_files:
    print(f, "✅" if os.path.exists(f) else "❌")

01_hoh_total_detections_by_hour.png ✅
01_hoh_total_detections_by_hour.html ✅
02_hoh_heatmap_top_species.png ✅
02_hoh_heatmap_top_species.html ✅
03_hoh_species_list_top20.png ✅
03_hoh_species_list_full.csv ✅
04_hoh_top10_species_bar.png ✅
04_hoh_top10_species_bar.html ✅
04_hoh_top10_species_table.png ✅
04_hoh_top10_species_table.csv ✅


In [ ]:
# ============================================
# CELL 10 — Optional: create a zip file for download
# ============================================
import zipfile

zip_filename = "hoh_presentation_outputs.zip"

with zipfile.ZipFile(zip_filename, "w") as zipf:
    for f in expected_files:
        if os.path.exists(f):
            zipf.write(f)

print(f"Created zip: {zip_filename}")

Created zip: hoh_presentation_outputs.zip


In [ ]:
# ============================================
# CELL 6 — Figure 5: Species richness by hour
# ============================================
richness_by_hour = (
    df[df["detections"] > 0]
    .groupby("hour")["species"]
    .nunique()
    .reset_index(name="species_richness")
    .sort_values("hour")
)

all_hours = pd.DataFrame({"hour": list(range(24))})
richness_by_hour = all_hours.merge(richness_by_hour, on="hour", how="left").fillna(0)
richness_by_hour["species_richness"] = richness_by_hour["species_richness"].astype(int)

peak_row = richness_by_hour.loc[richness_by_hour["species_richness"].idxmax()]
peak_hour = int(peak_row["hour"])
peak_value = int(peak_row["species_richness"])

fig5 = go.Figure()

fig5.add_trace(go.Scatter(
    x=richness_by_hour["hour"],
    y=richness_by_hour["species_richness"],
    mode="lines+markers",
    line=dict(width=4, shape="spline", smoothing=1.0),
    marker=dict(size=8),
    fill="tozeroy",
    hovertemplate="<b>%{x}:00</b><br>Species richness: %{y}<extra></extra>",
    name="Species richness"
))

fig5.add_trace(go.Scatter(
    x=[peak_hour],
    y=[peak_value],
    mode="markers+text",
    marker=dict(size=16, symbol="star"),
    text=[f"Peak: {peak_hour}:00"],
    textposition="top center",
    hovertemplate=f"<b>Peak richness</b><br>{peak_hour}:00<br>Species: {peak_value}<extra></extra>",
    name="Peak"
))

fig5.update_layout(
    title=dict(
        text="Species Richness Across the Day at Hoh",
        x=0.5,
        xanchor="center"
    ),
    xaxis=dict(
        title="Hour of day",
        tickmode="array",
        tickvals=list(range(24)),
        ticktext=[f"{h}:00" for h in range(24)],
        showgrid=True
    ),
    yaxis=dict(
        title="Number of species detected",
        showgrid=True
    ),
    template="plotly_white",
    height=520,
    width=1000,
    hovermode="x unified"
)

fig5.add_annotation(
    x=peak_hour,
    y=peak_value,
    text=f"Highest richness at {peak_hour}:00<br>{peak_value} species detected",
    showarrow=True,
    arrowhead=2,
    ax=0,
    ay=-60
)

fig5.show()

export_plotly(
    fig5,
    "05_hoh_species_richness_by_hour.png",
    "05_hoh_species_richness_by_hour.html",
    width=1500,
    height=850,
    scale=3
)

Saved: 05_hoh_species_richness_by_hour.png
Saved: 05_hoh_species_richness_by_hour.html


In [ ]:
# ============================================
# CELL 8A — Figure 6: Representative species activity curves (manual selection)
# ============================================
representative_species = [
    "Marbled Murrelet",
    "Pacific Wren",
    "Varied Thrush",
    "Swainson's Thrush"
]

df_rep = df[df["species"].isin(representative_species)].copy()

hour_species = (
    df_rep.groupby(["species", "hour"], as_index=False)["detections"]
          .sum()
)

all_hours = pd.DataFrame({"hour": list(range(24))})
full_grid = pd.MultiIndex.from_product(
    [representative_species, range(24)],
    names=["species", "hour"]
).to_frame(index=False)

hour_species = full_grid.merge(hour_species, on=["species", "hour"], how="left").fillna(0)

fig6 = go.Figure()

for sp in representative_species:
    sub = hour_species[hour_species["species"] == sp].sort_values("hour")
    fig6.add_trace(go.Scatter(
        x=sub["hour"],
        y=sub["detections"],
        mode="lines+markers",
        name=sp,
        hovertemplate=f"<b>{sp}</b><br>Hour: %{{x}}:00<br>Detections: %{{y:,.0f}}<extra></extra>"
    ))

fig6.update_layout(
    title=dict(
        text="Activity Patterns of Representative Species at Hoh",
        x=0.5,
        xanchor="center"
    ),
    xaxis=dict(
        title="Hour of day",
        tickmode="array",
        tickvals=list(range(24)),
        ticktext=[f"{h}:00" for h in range(24)],
        showgrid=True
    ),
    yaxis=dict(
        title="Detections per hour",
        showgrid=True,
        separatethousands=True
    ),
    template="plotly_white",
    height=600,
    width=1100,
    hovermode="x unified",
    legend=dict(
        title="Species",
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    )
)

fig6.show()

export_plotly(
    fig6,
    "06_hoh_representative_species_activity_curves.png",
    "06_hoh_representative_species_activity_curves.html",
    width=1600,
    height=900,
    scale=3
)

Saved: 06_hoh_representative_species_activity_curves.png
Saved: 06_hoh_representative_species_activity_curves.html


In [ ]:
import zipfile

files_to_zip = [
    "05_hoh_species_richness_by_hour.png",
    "05_hoh_species_richness_by_hour.html",
    "06_hoh_representative_species_activity_curves.png",
    "06_hoh_representative_species_activity_curves.html"
]

zip_name = "hoh_figures_3_4.zip"

with zipfile.ZipFile(zip_name, "w") as z:
    for f in files_to_zip:
        z.write(f)

print("Zip file created:", zip_name)